## 1. Imports and shared configuration

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

# Paths
TRAIN_NEWS_PATH = "/kaggle/input/datasets/teshanlakruwan/mind-small-train/news.tsv"
TRAIN_BEHAVIORS_PATH = "/kaggle/input/datasets/teshanlakruwan/mind-small-train/behaviors.tsv"
DEV_NEWS_PATH = "/kaggle/input/datasets/teshanlakruwan/mind-small-dev/news.tsv"
DEV_BEHAVIORS_PATH = "/kaggle/input/datasets/teshanlakruwan/mind-small-dev/behaviors.tsv"
ADS_POOL_PATH = "/kaggle/input/datasets/teshanlakruwan/ad-pool/ads_pool.csv"
WORKDIR = "/kaggle/working"

RANDOM_STATE = 42
TRAIN_SAMPLE_SIZE = 200000
DEV_SAMPLE_SIZE = 50000
STAGE2_TRAIN_SAMPLE = 200000  # used during Stage 2 training for faster iteration

news_columns = [
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities",
]

behaviors_columns = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

mind_to_iab = {
    "sports": "Sports",
    "finance": "Business_Finance",
    "autos": "Automotive",
    "travel": "Travel",
    "health": "Health",
    "lifestyle": "Lifestyle",
    "foodanddrink": "Food_Drink",
    "news": "General_News",
    "weather": "General_News",
    "middleeast": "General_News",
    "northamerica": "General_News",
    "entertainment": "Entertainment",
    "tv": "Entertainment",
    "movies": "Entertainment",
    "music": "Entertainment",
    "video": "Entertainment",
    "kids": "Lifestyle",
}

## 2. Helper functions

In [2]:
def map_to_iab(category: str) -> str:
    return mind_to_iab.get(str(category).lower().strip(), "Other")

def parse_history(history):
    if pd.isna(history) or history == "":
        return []
    return str(history).split()

def parse_impressions(impressions):
    result = []
    if pd.isna(impressions):
        return result
    for item in str(impressions).split():
        news_id, clicked = item.split("-")
        result.append((news_id, int(clicked)))
    return result

def build_master_dataset(news_path, behaviors_path, sample_size=None):
    news_df = pd.read_csv(
        news_path,
        sep="\t",
        header=None,
        names=news_columns,
    )
    behaviors_df = pd.read_csv(
        behaviors_path,
        sep="\t",
        header=None,
        names=behaviors_columns,
    )

    # Keep required news fields
    news_df = news_df[["news_id", "category", "subcategory", "title", "abstract"]].copy()
    news_df["category_lower"] = news_df["category"].str.lower().str.strip()
    news_df["iab_category"] = news_df["category_lower"].apply(map_to_iab)
    news_df = news_df[news_df["iab_category"] != "Other"].copy()

    # Behaviors
    behaviors_df["history_list"] = behaviors_df["history"].apply(parse_history)
    behaviors_df["history_count"] = behaviors_df["history_list"].apply(len)
    behaviors_df["impression_items"] = behaviors_df["impressions"].apply(parse_impressions)

    rows = []
    for _, row in behaviors_df.iterrows():
        for news_id, clicked in row["impression_items"]:
            rows.append({
                "impression_id": row["impression_id"],
                "user_id": row["user_id"],
                "time": row["time"],
                "history_list": row["history_list"],
                "history_count": row["history_count"],
                "candidate_news_id": news_id,
                "clicked": clicked,
            })

    expanded_df = pd.DataFrame(rows)

    master_df = expanded_df.merge(
        news_df,
        left_on="candidate_news_id",
        right_on="news_id",
        how="inner",
    )

    master_df["history_unique_count"] = master_df["history_list"].apply(lambda x: len(set(x)))
    master_df["current_in_history"] = master_df.apply(
        lambda row: 1 if row["candidate_news_id"] in row["history_list"] else 0,
        axis=1,
    )

    for col in ["title", "abstract", "subcategory"]:
        master_df[col] = master_df[col].fillna("").astype(str)

    master_df["page_text"] = (
        master_df["title"] + " " + master_df["abstract"] + " " + master_df["subcategory"]
    ).str.strip()

    if sample_size is not None and len(master_df) > sample_size:
        master_df = master_df.sample(n=sample_size, random_state=RANDOM_STATE).reset_index(drop=True)
    else:
        master_df = master_df.reset_index(drop=True)

    return master_df

def tokenize_text(text):
    if pd.isna(text):
        return []
    return str(text).lower().split()

def simple_overlap_similarity(text1, text2):
    set1 = set(tokenize_text(text1))
    set2 = set(tokenize_text(text2))
    if len(set1) == 0 or len(set2) == 0:
        return 0.0
    return len(set1.intersection(set2)) / len(set1.union(set2))

def print_weighted_metrics(y_true, y_pred, title="Results", probs=None):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )
    print(f"=== {title} ===")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1-score :", f1)
    if probs is not None and len(np.unique(y_true)) > 1:
        print("ROC-AUC  :", roc_auc_score(y_true, probs))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nDetailed Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

## 3. Build train and dev master datasets

In [3]:
master_train_df = build_master_dataset(
    TRAIN_NEWS_PATH,
    TRAIN_BEHAVIORS_PATH,
    sample_size=TRAIN_SAMPLE_SIZE,
)

master_dev_df = build_master_dataset(
    DEV_NEWS_PATH,
    DEV_BEHAVIORS_PATH,
    sample_size=DEV_SAMPLE_SIZE,
)

print("Train master shape:", master_train_df.shape)
print("Dev master shape  :", master_dev_df.shape)

master_train_df.to_csv(f"{WORKDIR}/master_train_dataset.csv", index=False)
master_dev_df.to_csv(f"{WORKDIR}/master_dev_dataset.csv", index=False)
print("Saved master datasets to /kaggle/working")

Train master shape: (200000, 17)
Dev master shape  : (50000, 17)
Saved master datasets to /kaggle/working


## 4. Stage 1 — contextual category prediction

In [4]:
# Labels
y_train = master_train_df["iab_category"]
y_dev = master_dev_df["iab_category"]

# Text
X_train_text_raw = master_train_df["page_text"]
X_dev_text_raw = master_dev_df["page_text"]

# Behaviour
behaviour_cols = ["history_count", "history_unique_count", "current_in_history"]
X_train_behaviour = master_train_df[behaviour_cols].copy()
X_dev_behaviour = master_dev_df[behaviour_cols].copy()

# TF-IDF
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
)

X_train_text = tfidf.fit_transform(X_train_text_raw)
X_dev_text = tfidf.transform(X_dev_text_raw)

# Encode labels
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_dev_enc = label_encoder.transform(y_dev)

print("TF-IDF train shape:", X_train_text.shape)
print("TF-IDF dev shape  :", X_dev_text.shape)
print("Classes:", list(label_encoder.classes_))

TF-IDF train shape: (200000, 5000)
TF-IDF dev shape  : (50000, 5000)
Classes: ['Automotive', 'Business_Finance', 'Entertainment', 'Food_Drink', 'General_News', 'Health', 'Lifestyle', 'Sports', 'Travel']


## 5. Stage 1 evaluation: text-only, behaviour-only, and combined baselines

In [5]:
# Text-only
text_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
text_model.fit(X_train_text, y_train_enc)
y_pred_text = text_model.predict(X_dev_text)

# Behaviour-only
standard_scaler = StandardScaler()
X_train_behaviour_scaled = standard_scaler.fit_transform(X_train_behaviour)
X_dev_behaviour_scaled = standard_scaler.transform(X_dev_behaviour)

behaviour_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
behaviour_model.fit(X_train_behaviour_scaled, y_train_enc)
y_pred_behaviour = behaviour_model.predict(X_dev_behaviour_scaled)

# Combined baseline
X_train_combined = hstack([X_train_text, csr_matrix(X_train_behaviour_scaled)])
X_dev_combined = hstack([X_dev_text, csr_matrix(X_dev_behaviour_scaled)])

combined_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
combined_model.fit(X_train_combined, y_train_enc)
y_pred_combined = combined_model.predict(X_dev_combined)

# Summary table
results = []

def add_result(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1_score": f1,
    })

add_result("Text Only", y_dev_enc, y_pred_text)
add_result("Behaviour Only", y_dev_enc, y_pred_behaviour)
add_result("Text + Behaviour", y_dev_enc, y_pred_combined)

results_df = pd.DataFrame(results)
print(results_df)
print("\nDetailed report: Text Only")
print(classification_report(y_dev_enc, y_pred_text, target_names=label_encoder.classes_, zero_division=0))

              Model  Accuracy  Precision   Recall  F1_score
0         Text Only   0.95960   0.961298  0.95960  0.959043
1    Behaviour Only   0.15558   0.127901  0.15558  0.080490
2  Text + Behaviour   0.95740   0.958918  0.95740  0.956849

Detailed report: Text Only
                  precision    recall  f1-score   support

      Automotive       0.91      0.96      0.94      2024
Business_Finance       1.00      1.00      1.00      4096
   Entertainment       0.98      0.99      0.98      9779
      Food_Drink       0.95      1.00      0.97      3863
    General_News       0.92      1.00      0.96     12533
          Health       1.00      0.94      0.97      2478
       Lifestyle       0.98      0.92      0.95      6505
          Sports       0.97      0.91      0.94      6410
          Travel       1.00      0.77      0.87      2312

        accuracy                           0.96     50000
       macro avg       0.97      0.94      0.95     50000
    weighted avg       0.96      0

## 6. Save Stage 1 artifacts and generate Stage 1 outputs

In [6]:
# Save core Stage 1 artifacts
joblib.dump(tfidf, f"{WORKDIR}/tfidf_vectorizer.pkl")
joblib.dump(text_model, f"{WORKDIR}/stage1_model.pkl")
joblib.dump(label_encoder, f"{WORKDIR}/label_encoder.pkl")
print("Saved Stage 1 artifacts")

# Train predictions
train_pred = text_model.predict(X_train_text)
train_prob = text_model.predict_proba(X_train_text)
dev_pred = text_model.predict(X_dev_text)
dev_prob = text_model.predict_proba(X_dev_text)

master_train_df["predicted_category"] = label_encoder.inverse_transform(train_pred)
master_train_df["stage1_confidence"] = train_prob.max(axis=1)

master_dev_df["predicted_category"] = label_encoder.inverse_transform(dev_pred)
master_dev_df["stage1_confidence"] = dev_prob.max(axis=1)

master_train_df.to_csv(f"{WORKDIR}/train_with_stage1_outputs.csv", index=False)
master_dev_df.to_csv(f"{WORKDIR}/dev_with_stage1_outputs.csv", index=False)
print("Saved Stage 1 outputs")

Saved Stage 1 artifacts
Saved Stage 1 outputs


## 7. Build normalized behaviour score

In [7]:
train_df = pd.read_csv(f"{WORKDIR}/train_with_stage1_outputs.csv")
dev_df = pd.read_csv(f"{WORKDIR}/dev_with_stage1_outputs.csv")

behaviour_scaler = MinMaxScaler()
train_df[behaviour_cols] = behaviour_scaler.fit_transform(train_df[behaviour_cols])
dev_df[behaviour_cols] = behaviour_scaler.transform(dev_df[behaviour_cols])

train_df["behaviour_score"] = (
    0.4 * train_df["history_count"] +
    0.3 * train_df["history_unique_count"] +
    0.3 * train_df["current_in_history"]
)

dev_df["behaviour_score"] = (
    0.4 * dev_df["history_count"] +
    0.3 * dev_df["history_unique_count"] +
    0.3 * dev_df["current_in_history"]
)

joblib.dump(behaviour_scaler, f"{WORKDIR}/behaviour_scaler.pkl")
train_df.to_csv(f"{WORKDIR}/train_with_behaviour_score.csv", index=False)
dev_df.to_csv(f"{WORKDIR}/dev_with_behaviour_score.csv", index=False)
print("Saved behaviour-enriched train/dev datasets")

Saved behaviour-enriched train/dev datasets


## 8. Stage 2 — build page–ad pair dataset

This stage treats ad selection as a **ranking / suitability problem**.  
Each row is a `(page, ad)` pair with:
- Stage 1 confidence
- behaviour score
- overlap similarity
- category match
- ad type
- proxy suitability label

In [ ]:
train_df = pd.read_csv(f"{WORKDIR}/train_with_behaviour_score.csv")
ads_df = pd.read_csv(ADS_POOL_PATH)

train_df = train_df.reset_index(drop=True)
ads_df = ads_df.reset_index(drop=True)

ads_df["ad_text"] = ads_df["ad_text"].fillna("").astype(str)
ads_df["category"] = ads_df["category"].fillna("Unknown").astype(str)
ads_df["type"] = ads_df["type"].fillna("generic").astype(str)

same_cat_n = 3
other_cat_n = 2
generic_n = 1

generic_ads = ads_df[ads_df["type"].str.lower() == "generic"]

pairs = []

for i, row in train_df.iterrows():
    page_cat = row["predicted_category"]
    page_text = row["page_text"]

    same_cat_ads = ads_df[ads_df["category"] == page_cat]
    other_cat_ads = ads_df[ads_df["category"] != page_cat]

    sampled_same = (
        same_cat_ads.sample(n=min(same_cat_n, len(same_cat_ads)), random_state=RANDOM_STATE)
        if len(same_cat_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
    )
    sampled_other = (
        other_cat_ads.sample(n=min(other_cat_n, len(other_cat_ads)), random_state=RANDOM_STATE)
        if len(other_cat_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
    )
    sampled_generic = (
        generic_ads.sample(n=min(generic_n, len(generic_ads)), random_state=RANDOM_STATE)
        if len(generic_ads) > 0 else pd.DataFrame(columns=ads_df.columns)
    )

    candidate_ads = pd.concat(
        [sampled_same, sampled_other, sampled_generic],
        ignore_index=True,
    ).drop_duplicates(subset=["ad_id"])

    for _, ad_row in candidate_ads.iterrows():
        text_similarity = simple_overlap_similarity(page_text, ad_row["ad_text"])
        category_match = int(page_cat == ad_row["category"])
        ad_type_targeted = int(str(ad_row["type"]).lower() == "targeted")

        # Final proxy label used for the prototype
        label = int(
            (row["clicked"] == 1) and
            (
                category_match == 1 or
                text_similarity > 0.05
            )
        )

        pairs.append({
            "page_id": i,
            "candidate_news_id": row["candidate_news_id"],
            "ad_id": ad_row["ad_id"],
            "page_category": page_cat,
            "ad_category": ad_row["category"],
            "stage1_confidence": float(row["stage1_confidence"]),
            "behaviour_score": float(row["behaviour_score"]),
            "text_similarity": float(text_similarity),
            "category_match": int(category_match),
            "ad_type_targeted": int(ad_type_targeted),
            "label": int(label),
        })

pair_df = pd.DataFrame(pairs)
pair_df.to_csv(f"{WORKDIR}/stage2_train_pairs_final.csv", index=False)

print("Saved:", f"{WORKDIR}/stage2_train_pairs_final.csv")
print("Shape:", pair_df.shape)
print(pair_df.head())
print("\nLabel distribution:")
print(pair_df["label"].value_counts(normalize=True))
print("\nZero similarity ratio:")
print((pair_df["text_similarity"] == 0).mean())

## 9. Train final Stage 2 suitability model

In [ ]:
pair_df = pd.read_csv(f"{WORKDIR}/stage2_train_pairs_final.csv")

# Sample for practical training time
pair_df = pair_df.sample(n=min(STAGE2_TRAIN_SAMPLE, len(pair_df)), random_state=RANDOM_STATE).reset_index(drop=True)

stage2_features = [
    "stage1_confidence",
    "behaviour_score",
    "text_similarity",
    "category_match",
    "ad_type_targeted",
]

X = pair_df[stage2_features].fillna(0)
y = pd.to_numeric(pair_df["label"], errors="coerce").fillna(0).astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

stage2_model = LogisticRegression(
    max_iter=300,
    random_state=RANDOM_STATE,
    solver="saga",
    n_jobs=-1,
)

stage2_model.fit(X_train, y_train)

y_pred = stage2_model.predict(X_val)
y_prob = stage2_model.predict_proba(X_val)[:, 1]

print_weighted_metrics(y_val, y_pred, title="Final Stage 2 Results", probs=y_prob)

coef_df = pd.DataFrame({
    "feature": stage2_features,
    "coefficient": stage2_model.coef_[0],
}).sort_values(by="coefficient", ascending=False)

print("\nFeature Coefficients:")
print(coef_df)

joblib.dump(stage2_model, f"{WORKDIR}/stage2_suitability_model_final.pkl")
print("\nSaved:", f"{WORKDIR}/stage2_suitability_model_final.pkl")

## 10. Final inference demo: rank ads for a new page

At inference time, candidate ads are restricted to:
- same predicted category ads
- generic ads

This improves real-world plausibility.

In [ ]:
stage2_model = joblib.load(f"{WORKDIR}/stage2_suitability_model_final.pkl")
dev_df = pd.read_csv(f"{WORKDIR}/dev_with_behaviour_score.csv")
ads_df = pd.read_csv(ADS_POOL_PATH)

ads_df["ad_text"] = ads_df["ad_text"].fillna("").astype(str)
ads_df["category"] = ads_df["category"].fillna("Unknown").astype(str)
ads_df["type"] = ads_df["type"].fillna("generic").astype(str)

page_idx = 0
page_row = dev_df.iloc[page_idx]

page_text = str(page_row["page_text"])
predicted_category = str(page_row["predicted_category"])
stage1_confidence = float(page_row["stage1_confidence"])
behaviour_score = float(page_row["behaviour_score"])

same_cat_ads = ads_df[ads_df["category"] == predicted_category]
generic_ads = ads_df[ads_df["type"].str.lower() == "generic"]

candidate_ads = pd.concat(
    [same_cat_ads, generic_ads],
    ignore_index=True,
).drop_duplicates(subset=["ad_id"]).reset_index(drop=True)

rank_rows = []
for _, ad_row in candidate_ads.iterrows():
    text_similarity = simple_overlap_similarity(page_text, ad_row["ad_text"])
    category_match = int(predicted_category == ad_row["category"])
    ad_type_targeted = int(str(ad_row["type"]).lower() == "targeted")

    feature_row = pd.DataFrame([{
        "stage1_confidence": stage1_confidence,
        "behaviour_score": behaviour_score,
        "text_similarity": text_similarity,
        "category_match": category_match,
        "ad_type_targeted": ad_type_targeted,
    }])

    suitability_score = stage2_model.predict_proba(feature_row)[0][1]

    rank_rows.append({
        "ad_id": ad_row["ad_id"],
        "ad_category": ad_row["category"],
        "ad_type": ad_row["type"],
        "text_similarity": text_similarity,
        "category_match": category_match,
        "suitability_score": suitability_score,
        "ad_text": ad_row["ad_text"],
    })

ranked_ads = pd.DataFrame(rank_rows).sort_values(
    by="suitability_score",
    ascending=False,
).reset_index(drop=True)

final_ad = ranked_ads.iloc[0]

print("=== NEW PAGE ===")
print("Page index:", page_idx)
print("Predicted category:", predicted_category)
print("Stage 1 confidence:", stage1_confidence)
print("Behaviour score:", behaviour_score)
print("\nPage text:")
print(page_text[:500])

print("\n=== TOP RANKED ADS ===")
print(ranked_ads.head(10))

print("\n=== FINAL SELECTED AD ===")
print(final_ad)